# Evidence-First Failure Analysis (Categories A, B, C, D)
### Project: Agentic Multimodal Retrieval Controller for KB-VQA
**Document Reference:**  (Phase 1)

This notebook processes the outputs of **Step 3** ( / ) alongside **Step 2** () to classify every test query into our 4-way Failure Taxonomy:
- **Correct:** VQA prediction matches ground truth.
- **Category A (Retrieval Ceiling Failure):** Correct entity omitted in Step 1 coarse search (never in top-20).
- **Category B (Step 2 Reranker Failure):** Correct entity was in top-20, but Step 2 ranked it > 1.
- **Category C (Section Selection Failure):** Correct entity retrieved, but chosen section lacked evidence.
- **Category D (Generator Reasoning Failure):** Correct entity + evidence present in section, but generator answered incorrectly.

It also computes the **confidence margin distributions ( - S_2$)** to demonstrate the mathematical trigger $	au$ for our Agent Controller.

In [ ]:
import os
import sys
import json
import re
import string
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully.")

## 1. Configure File Paths
Automatically searches standard Kaggle directories or local fallback paths.

In [ ]:
KAGGLE_DS_DIR = "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds"

def find_file(candidates, name):
    for c in candidates:
        if os.path.exists(c):
            print(f"[Found] {name}: {c}")
            return c
    print(f"[Warning] Could not find {name} in {candidates}")
    return candidates[0]

STEP3_FILE = find_file([
    "/kaggle/working/step3_results/step3_final_vqa_results.json",
    "/kaggle/working/results_step_3.json",
    os.path.join(KAGGLE_DS_DIR, "results_step_3.json"),
    os.path.join(KAGGLE_DS_DIR, "step3_final_vqa_results.json"),
    "pre helper/res/results_step_3.json"
], "Step 3 Result")

STEP2_FILE = find_file([
    os.path.join(KAGGLE_DS_DIR, "result_step2.json"),
    os.path.join(KAGGLE_DS_DIR, "res_step2.json"),
    "pre helper/res/result_step2.json"
], "Step 2 Result")

SAMPLE_FILE = find_file([
    os.path.join(KAGGLE_DS_DIR, "infoseek_test_sample_1000.csv"),
    "pre helper/files/infoseek_test_sample_1000.csv"
], "Sample CSV")

KB_FILE = find_file([
    os.path.join(KAGGLE_DS_DIR, "infoseek_100k_wiki/wiki_100_dict_v4.json"),
    "/kaggle/input/infoseek-100k-wiki/wiki_100_dict_v4.json",
    "pre helper/files/wiki_100_dict_v4.json"
], "Wiki KB JSON")

OUTPUT_DIR = "/kaggle/working/failure_analysis_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. InfoSeek Evaluation Metric Helpers
Implements official exact match and numerical range matching.

In [ ]:
def normalize_answer(s):
    if not s: return ""
    def remove_articles(text): return re.sub(r"(a|an|the)", " ", text)
    def white_space_fix(text): return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(str(s).lower())))

def exact_match_score(pred, gt):
    return normalize_answer(pred) == normalize_answer(gt)

def extract_number(val):
    if isinstance(val, (int, float)): return float(val)
    if not isinstance(val, str): return None
    val = val.replace(",", "").strip()
    m = re.search(r"[-+]?\d*\.?\d+", val)
    return float(m.group()) if m else None

def metric_numerical_range(pred_num, gt_num, tol=0.05):
    if abs(gt_num) < 1e-6: return abs(pred_num - gt_num) < 1e-4
    return abs(pred_num - gt_num) / abs(gt_num) <= tol

def evaluate_prediction(pred_str, gt_answers):
    if not pred_str: return False
    norm_pred = normalize_answer(pred_str)
    for gt in gt_answers:
        if exact_match_score(pred_str, gt): return True
        norm_gt = normalize_answer(gt)
        if norm_gt and (norm_gt == norm_pred or f" {norm_gt} " in f" {norm_pred} "):
            return True
    pred_val = extract_number(pred_str)
    if pred_val is not None:
        for gt in gt_answers:
            gt_val = extract_number(gt)
            if gt_val is not None and metric_numerical_range(pred_val, gt_val):
                return True
    return False

## 3. Load Datasets & Step 2/3 Results

In [ ]:
# Load Samples
df_samples = pd.read_csv(SAMPLE_FILE)
samples_dict = {}
for _, row in df_samples.iterrows():
    did = str(row.get("data_id", ""))
    samples_dict[did] = {
        "data_id": did,
        "question": str(row.get("question", "")),
        "image_id": str(row.get("dataset_image_ids", "")).split("|")[0],
        "wikipedia_url": str(row.get("wikipedia_url", "")).strip(),
        "answers": [a.strip() for a in str(row.get("answer", "")).split("|") if a.strip()],
    }
print(f"Loaded {len(samples_dict)} sample queries.")

# Load Step 2
with open(STEP2_FILE, "r", encoding="utf-8") as f:
    step2_dict = json.load(f)
print(f"Loaded {len(step2_dict)} Step 2 queries.")

# Load Step 3
with open(STEP3_FILE, "r", encoding="utf-8") as f:
    raw_step3 = json.load(f)

step3_dict = {}
if isinstance(raw_step3, list):
    for item in raw_step3:
        step3_dict[str(item.get("data_id", ""))] = item
else:
    for k, v in raw_step3.items():
        step3_dict[str(k)] = v if isinstance(v, dict) else {"data_id": str(k), "prediction": str(v)}
print(f"Loaded {len(step3_dict)} Step 3 queries.")

# Optional KB load
kb_dict = {}
if os.path.exists(KB_FILE):
    print(f"Loading KB from {KB_FILE}...")
    with open(KB_FILE, "r", encoding="utf-8") as f:
        raw_kb = json.load(f)
    for u, inf in raw_kb.items():
        kb_dict[u] = {
            "title": inf.get("title", ""),
            "section_titles": inf.get("section_titles", []),
            "section_texts": inf.get("section_texts", []),
        }
    print(f"KB loaded with {len(kb_dict)} articles.")

## 4. Run Failure Classification & Margin Analysis

In [ ]:
cat_a, cat_b, cat_c, cat_d, correct = [], [], [], [], []
margins_correct, margins_cat_a, margins_cat_b, margins_cat_c, margins_cat_d = [], [], [], [], []

for idx, (did, s_meta) in enumerate(samples_dict.items()):
    q_text = s_meta["question"]
    gt_url = s_meta["wikipedia_url"]
    gt_ans = s_meta["answers"]
    img_id = s_meta["image_id"]

    s2_entry = step2_dict.get(did) or step2_dict.get(str(did)) or (list(step2_dict.values())[idx] if idx < len(step2_dict) else {})
    cands = [e.get("url", "").strip() for e in s2_entry.get("retrieved_entities", [])]
    scores = s2_entry.get("retrieval_similarities", [])
    top1_url = cands[0] if len(cands) > 0 else ""
    margin = float(scores[0] - scores[1]) if len(scores) > 1 else 0.0

    s3_entry = step3_dict.get(did) or step3_dict.get(str(did)) or (list(step3_dict.values())[idx] if idx < len(step3_dict) else {})
    pred = s3_entry.get("prediction", "")
    chosen_sec = s3_entry.get("top1_section") or s3_entry.get("selected_section") or s3_entry.get("section_text", "")

    is_corr = s3_entry.get("is_correct")
    if is_corr is None:
        is_corr = evaluate_prediction(pred, gt_ans)

    gt_in_top20 = gt_url in cands[:20]
    gt_rank = cands.index(gt_url) + 1 if gt_in_top20 else -1

    item = {
        "data_id": did,
        "question": q_text,
        "ground_truth_url": gt_url,
        "ground_truth_answers": gt_ans,
        "prediction": pred,
        "is_correct": bool(is_corr),
        "top1_url": top1_url,
        "score_margin": margin,
        "gt_rank": gt_rank,
    }

    if is_corr:
        correct.append(item)
        margins_correct.append(margin)
        continue

    if not gt_in_top20:
        cat_a.append(item)
        margins_cat_a.append(margin)
    elif top1_url != gt_url:
        cat_b.append(item)
        margins_cat_b.append(margin)
    else:
        # Top-1 entity was correct, check section
        evidence_in_chosen = any(normalize_answer(a) in normalize_answer(chosen_sec) for a in gt_ans) if chosen_sec else False
        if not evidence_in_chosen:
            cat_c.append(item)
            margins_cat_c.append(margin)
        else:
            cat_d.append(item)
            margins_cat_d.append(margin)

total_q = len(samples_dict)
print("=" * 60)
print("FAILURE TAXONOMY SUMMARY")
print("=" * 60)
print(f"Correct VQA Predictions   : {len(correct):4d} ({len(correct)/total_q*100:5.2f}%)  [Mean Margin: {np.mean(margins_correct):.4f}]")
print(f"Category A (Ceiling Error): {len(cat_a):4d} ({len(cat_a)/total_q*100:5.2f}%)  [Mean Margin: {np.mean(margins_cat_a):.4f}]")
print(f"Category B (Rerank Error) : {len(cat_b):4d} ({len(cat_b)/total_q*100:5.2f}%)  [Mean Margin: {np.mean(margins_cat_b):.4f}]")
print(f"Category C (Section Error): {len(cat_c):4d} ({len(cat_c)/total_q*100:5.2f}%)  [Mean Margin: {np.mean(margins_cat_c):.4f}]")
print(f"Category D (Gen Error)    : {len(cat_d):4d} ({len(cat_d)/total_q*100:5.2f}%)  [Mean Margin: {np.mean(margins_cat_d):.4f}]")
print("=" * 60)
ratio = np.mean(margins_correct) / (np.mean(margins_cat_b) + 1e-8)
print(f"Margin Drop Signal (Correct vs Category B): {ratio:.2f}x collapse!")

## 5. Visualizations & Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart of Breakdown
labels = ["Correct", "Cat A (Ceiling)", "Cat B (Rerank)", "Cat C (Section)", "Cat D (Gen)"]
counts = [len(correct), len(cat_a), len(cat_b), len(cat_c), len(cat_d)]
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#3498db", "#9b59b6"]

axes[0].bar(labels, counts, color=colors, edgecolor="black", linewidth=1.2)
axes[0].set_title("InfoSeek Baseline Failure Taxonomy", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Queries (out of 1,000)")
axes[0].tick_params(axis="x", rotation=25)
for i, v in enumerate(counts):
    axes[0].text(i, v + 10, f"{v}
({v/total_q*100:.1f}%)", ha="center", fontweight="bold")

# Margin Distribution
sns.kdeplot(margins_correct, ax=axes[1], label="Correct", color="#2ecc71", fill=True, alpha=0.3)
sns.kdeplot(margins_cat_b, ax=axes[1], label="Category B (Rerank Error)", color="#f39c12", fill=True, alpha=0.3)
axes[1].axvline(0.025, color="red", linestyle="--", label=r"Trigger Threshold $	au pprox 0.025$")
axes[1].set_title("Confidence Margin Collapse (S1 - S2)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Score Margin $\Delta = S_1 - S_2$")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "failure_distribution.png"), dpi=200)
plt.show()

## 6. Export Artifacts for Paper & Controller Construction

In [ ]:
summary = {
    "total_queries": total_q,
    "accuracy": round(len(correct) / total_q * 100, 2),
    "counts": {"correct": len(correct), "cat_a": len(cat_a), "cat_b": len(cat_b), "cat_c": len(cat_c), "cat_d": len(cat_d)},
    "margins": {
        "correct": float(np.mean(margins_correct)),
        "cat_b": float(np.mean(margins_cat_b)),
    },
}
with open(os.path.join(OUTPUT_DIR, "failure_taxonomy_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "failures_category_b.json"), "w") as f:
    json.dump(cat_b, f, indent=2)

print(f"All artifacts saved to {OUTPUT_DIR}/")